# Sprint 2b — Feature Engineering Notebook
## Changelog
| Version | Date | Changes |
|---------|------|---------|
| v1 | 2026-03-12 | Initial build: GTFS features (2b.1), ACS trend slopes (2b.2), feature consolidation (2b.3) |
| v2 | 2026-03-12 | Fix 4 critical bugs: (A) Census sentinel values not cleaned → NaN, (B) weekend/weekday ratio /5 error, (C) duplicate tract rows, (D) useless transit_desert interaction |
| v3 | 2026-03-15 | Modeling-ready cleanup: fix headsign fill bug, collapse multicollinear GTFS bands, drop redundant features (r=1.0, zero-var), drop weak/redundant interactions |

**Purpose:** Build the modeling-ready dataset for Sprint 2b predictive modeling.
Three tasks covered:
- **2b.1** GTFS feature engineering (headways, service span, frequency, weekend gap, route diversity)
- **2b.2** Multi-year ACS trend slopes (2019-2023 temporal changes per tract)
- **2b.3** Feature consolidation (merge all sources + interaction terms)

**Output:** `Sprint2b_Modeling_Features.csv` — one row per tract, all features + equity target.
This CSV is the input to Notebook 2 (Modeling & Risk Scoring).

## 1. Setup and Data Loading

Load all required libraries and data sources. The GTFS data contains 943K stop-time
records across 6,530 stops and 128 routes. We also load the Sprint 2a equity indicators
(our prediction target) and the multi-year ACS data (for temporal trends).

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

# Paths — adjust if your folder structure differs
BASE = '.'  # Run from Sprint 2b folder, or set to your project root
GTFS_PATH = f'{BASE}/Sprint 1 EDAs/transit_data.xlsx'
EQUITY_PATH = f'{BASE}/Sprint 2 /Composite Equity Indicators/Sprint2_Equity_Indicators_v3_tract.csv'
ACS_DIR = f'{BASE}/Sprint 2 /ACS Tract Level Data'
GPKG_PATH = f'{BASE}/MiamiDadeMpoAllAccessGpkg-expanded/12197701_au_2021_08.gpkg'
CROSSWALK_PATH = f'{BASE}/Sprint 2 /tab20_tract20_tract10_st12.txt'

print("Loading GTFS data...")
stop_times = pd.read_excel(GTFS_PATH, sheet_name='stop_times')
trips = pd.read_excel(GTFS_PATH, sheet_name='trips')
routes = pd.read_excel(GTFS_PATH, sheet_name='routes')
stops = pd.read_excel(GTFS_PATH, sheet_name='stops')
calendar = pd.read_excel(GTFS_PATH, sheet_name='calendar')

print(f"  stop_times: {stop_times.shape}")
print(f"  trips:      {trips.shape}")
print(f"  routes:     {routes.shape}")
print(f"  stops:      {stops.shape}")
print(f"  calendar:   {calendar.shape}")

print("\nLoading equity indicators (prediction target)...")
equity = pd.read_csv(EQUITY_PATH, dtype={'tract_geoid': str})
print(f"  equity: {equity.shape} — {equity['equity_tier'].value_counts().to_dict()}")

print("\nLoading multi-year ACS data...")
acs_years = {}
for year in [2019, 2020, 2021, 2022, 2023]:
    df = pd.read_csv(f'{ACS_DIR}/ACS_MiamiDade_Tracts_{year}.csv', dtype={'GEOID': str})
    acs_years[year] = df
    print(f"  {year}: {df.shape}")

# [v2 FIX A] Clean Census API sentinel values BEFORE any computation
# Census returns -666666666 for missing estimates and -999999999 for missing MOE.
# These are valid integers that slip through pd.to_numeric — they must be NaN.
CENSUS_SENTINELS = [-666666666, -666666666.0, -999999999, -999999999.0]
print("\nCleaning Census sentinel values (-666666666, -999999999) → NaN...")
for year, df in acs_years.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    sentinel_count = 0
    for col in numeric_cols:
        mask = df[col].isin(CENSUS_SENTINELS)
        n_found = mask.sum()
        if n_found > 0:
            df.loc[mask, col] = np.nan
            sentinel_count += n_found
    acs_years[year] = df
    print(f"  {year}: replaced {sentinel_count} sentinel values across {len(numeric_cols)} numeric columns")

print("\nAll data loaded and cleaned successfully.")

## 2. Spatial Join: Stops → Census Tracts

Before computing tract-level GTFS metrics, we need to know which tract each stop belongs to.
We load block-level geometries from the GeoPackage, dissolve them into tract boundaries,
then perform a point-in-polygon join using stop coordinates.

In [ ]:
# Load block geometries from GeoPackage (just the blocks table, not isochrones)
print("Loading block geometries from GeoPackage...")
blocks_gdf = gpd.read_file(GPKG_PATH, layer='blocks')
print(f"  Blocks loaded: {len(blocks_gdf)} rows")
print(f"  Columns: {list(blocks_gdf.columns)}")
print(f"  CRS: {blocks_gdf.crs}")

# Derive tract GEOID from blockid (first 11 digits of 15-digit block FIPS)
blocks_gdf['tract_geoid'] = blocks_gdf['blockid'].astype(str).str[:11]
n_tracts = blocks_gdf['tract_geoid'].nunique()
print(f"  Unique tracts in GeoPackage: {n_tracts}")

In [ ]:
# Dissolve blocks into tract boundaries
print("Dissolving blocks → tract boundaries...")
tract_boundaries = blocks_gdf.dissolve(by='tract_geoid').reset_index()[['tract_geoid', 'geometry']]
print(f"  Tract polygons: {len(tract_boundaries)}")

# Create GeoDataFrame of stops
stops_gdf = gpd.GeoDataFrame(
    stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
    geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
    crs='EPSG:4326'
)

# Ensure same CRS
if tract_boundaries.crs != stops_gdf.crs:
    tract_boundaries = tract_boundaries.to_crs(stops_gdf.crs)

# Spatial join: which tract does each stop fall in?
print("Spatial join: stops → tracts...")
stops_with_tract = gpd.sjoin(stops_gdf, tract_boundaries, how='left', predicate='within')

matched = stops_with_tract['tract_geoid'].notna().sum()
total = len(stops_with_tract)
print(f"  Matched: {matched}/{total} stops ({matched/total*100:.1f}%)")

# Handle unmatched stops (outside GeoPackage coverage) via nearest tract
unmatched_mask = stops_with_tract['tract_geoid'].isna()
if unmatched_mask.sum() > 0:
    print(f"  {unmatched_mask.sum()} unmatched stops — assigning to nearest tract...")
    from shapely.ops import nearest_points
    unmatched = stops_with_tract[unmatched_mask].copy()
    tract_unions = tract_boundaries.copy()
    
    for idx in unmatched.index:
        point = unmatched.loc[idx, 'geometry']
        distances = tract_boundaries.geometry.distance(point)
        nearest_idx = distances.idxmin()
        stops_with_tract.loc[idx, 'tract_geoid'] = tract_boundaries.loc[nearest_idx, 'tract_geoid']
    
    print(f"  All stops now assigned to tracts.")

# Clean up: keep the mapping
stop_tract_map = stops_with_tract[['stop_id', 'tract_geoid']].drop_duplicates(subset='stop_id')
print(f"\nStop → Tract mapping: {len(stop_tract_map)} stops across {stop_tract_map['tract_geoid'].nunique()} tracts")

## 3. GTFS Feature Engineering (Sub-Sprint 2b.1)

Compute granular transit service metrics from the GTFS schedule data.
These features serve dual purpose:
1. **Predictive features** for the ML model (what drives equity gaps?)
2. **Simulation levers** for Sprint 3 (what happens if we change headways?)

### Time bands used:
- **Peak AM**: 6:00–9:00
- **Midday**: 9:00–15:00
- **Peak PM**: 15:00–19:00
- **Evening**: 19:00–24:00
- **Early/Late**: 0:00–6:00 (overnight service)

### 3.1 Parse Times and Classify Service Days

GTFS times can exceed 24:00:00 for trips that cross midnight. We convert all times to
minutes-since-midnight for easy arithmetic, and classify each service_id as weekday,
Saturday, or Sunday using the calendar table.

In [ ]:
# Parse GTFS times to minutes since midnight
# GTFS allows times > 24:00:00 for trips spanning midnight
def parse_gtfs_time(time_str):
    """Convert HH:MM:SS string to minutes since midnight."""
    if pd.isna(time_str):
        return np.nan
    parts = str(time_str).split(':')
    return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60

stop_times['arrival_min'] = stop_times['arrival_time'].apply(parse_gtfs_time)
stop_times['departure_min'] = stop_times['departure_time'].apply(parse_gtfs_time)

print(f"Time range: {stop_times['arrival_min'].min():.0f} to {stop_times['arrival_min'].max():.0f} minutes")
print(f"  = {stop_times['arrival_min'].min()/60:.1f}h to {stop_times['arrival_min'].max()/60:.1f}h")
print(f"  Missing arrival times: {stop_times['arrival_min'].isna().sum()} ({stop_times['arrival_min'].isna().mean()*100:.1f}%)")

# Define time bands (in minutes since midnight)
TIME_BANDS = {
    'early':   (0, 360),      # 0:00 - 6:00
    'peak_am': (360, 540),    # 6:00 - 9:00
    'midday':  (540, 900),    # 9:00 - 15:00
    'peak_pm': (900, 1140),   # 15:00 - 19:00
    'evening': (1140, 1440),  # 19:00 - 24:00
}

def assign_time_band(minutes):
    """Assign a time-of-day band. Handles >24h GTFS times by wrapping."""
    if pd.isna(minutes):
        return None
    m = minutes % 1440  # wrap to 24h
    for band, (start, end) in TIME_BANDS.items():
        if start <= m < end:
            return band
    return 'early'  # fallback for edge cases

stop_times['time_band'] = stop_times['arrival_min'].apply(assign_time_band)
print(f"\nTrip distribution by time band:")
print(stop_times['time_band'].value_counts().sort_index())

In [ ]:
# Classify service_ids as weekday / saturday / sunday
weekday_services = calendar[
    (calendar['monday'] == 1) & (calendar['tuesday'] == 1) & 
    (calendar['wednesday'] == 1) & (calendar['thursday'] == 1) & 
    (calendar['friday'] == 1)
]['service_id'].tolist()

saturday_services = calendar[calendar['saturday'] == 1]['service_id'].tolist()
sunday_services = calendar[calendar['sunday'] == 1]['service_id'].tolist()

print(f"Weekday service_ids:  {weekday_services}")
print(f"Saturday service_ids: {saturday_services}")
print(f"Sunday service_ids:   {sunday_services}")

# Add service day type to trips
def classify_service(sid):
    if sid in weekday_services:
        return 'weekday'
    elif sid in saturday_services:
        return 'saturday'
    elif sid in sunday_services:
        return 'sunday'
    return 'other'

trips['service_day'] = trips['service_id'].apply(classify_service)
print(f"\nTrips by service day:\n{trips['service_day'].value_counts()}")

# Merge trip info (route_id, service_day) into stop_times
st = stop_times.merge(
    trips[['trip_id', 'route_id', 'service_id', 'service_day', 'direction_id', 
           'trip_headsign', 'wheelchair_accessible', 'bikes_allowed']],
    on='trip_id', how='left'
)

# Merge stop → tract mapping
st = st.merge(stop_tract_map, on='stop_id', how='left')

print(f"\nEnriched stop_times: {st.shape}")
print(f"  Tracts covered: {st['tract_geoid'].nunique()}")

### 3.2 Headway per Stop per Time Band

Headway = average minutes between consecutive vehicle arrivals at a stop.
Lower headway = more frequent service. We compute this for weekday service only
(weekday is the baseline; weekend comparison comes later).

In [ ]:
# Compute headways: for each stop × time band, sort arrivals and diff
# Focus on weekday service for baseline headways
weekday_st = st[st['service_day'] == 'weekday'].copy()

print("Computing headways per stop × time band (weekday service)...")
headways = []

for (stop_id, band), group in weekday_st.groupby(['stop_id', 'time_band']):
    arrivals = group['arrival_min'].dropna().sort_values()
    if len(arrivals) < 2:
        continue
    diffs = arrivals.diff().dropna()
    # Filter out unreasonable headways (> 120 min likely means service gap, not headway)
    diffs = diffs[(diffs > 0) & (diffs <= 120)]
    if len(diffs) == 0:
        continue
    headways.append({
        'stop_id': stop_id,
        'time_band': band,
        'mean_headway_min': diffs.mean(),
        'median_headway_min': diffs.median(),
        'n_arrivals': len(arrivals),
    })

headway_df = pd.DataFrame(headways)
print(f"  Headway records: {len(headway_df)} (stop × time_band combinations)")

# Merge tract info and aggregate to tract level
headway_df = headway_df.merge(stop_tract_map, on='stop_id', how='left')

# Pivot: one column per time band
tract_headways = headway_df.groupby(['tract_geoid', 'time_band'])['mean_headway_min'].mean().unstack(fill_value=np.nan)
tract_headways.columns = [f'headway_{band}_min' for band in tract_headways.columns]
tract_headways = tract_headways.reset_index()

print(f"  Tract-level headways: {tract_headways.shape}")
print(f"\nMedian headways by time band:")
for col in [c for c in tract_headways.columns if c.startswith('headway_')]:
    med = tract_headways[col].median()
    print(f"  {col}: {med:.1f} min")

### 3.3 Service Span per Stop

Service span = hours between first arrival and last departure at a stop.
Longer span means the stop is usable for more of the day — critical for
shift workers, evening commuters, and late-night travelers.

In [ ]:
# Service span: first arrival to last departure per stop (weekday)
print("Computing service span per stop (weekday)...")

span_df = weekday_st.groupby('stop_id').agg(
    first_arrival=('arrival_min', 'min'),
    last_departure=('departure_min', 'max'),
    n_trips=('trip_id', 'nunique')
).reset_index()

span_df['service_span_hours'] = (span_df['last_departure'] - span_df['first_arrival']) / 60
span_df = span_df.merge(stop_tract_map, on='stop_id', how='left')

# Aggregate to tract: mean and max service span
tract_span = span_df.groupby('tract_geoid').agg(
    mean_service_span_hours=('service_span_hours', 'mean'),
    max_service_span_hours=('service_span_hours', 'max'),
    total_weekday_trips=('n_trips', 'sum'),
).reset_index()

print(f"  Tract-level service span: {tract_span.shape}")
print(f"  Mean service span: {tract_span['mean_service_span_hours'].median():.1f} hours (median across tracts)")
print(f"  Max service span:  {tract_span['max_service_span_hours'].median():.1f} hours (median across tracts)")

### 3.4 Frequency by Time Band per Tract

Frequency = trips per hour arriving at stops within a tract, by time band.
This is a tract-level measure of how well-served an area is during different
parts of the day.

In [ ]:
# Frequency: trips per hour per tract per time band (weekday)
print("Computing frequency by time band per tract (weekday)...")

band_hours = {
    'early': 6, 'peak_am': 3, 'midday': 6, 'peak_pm': 4, 'evening': 5
}

freq_records = []
for (tract, band), group in weekday_st.groupby(['tract_geoid', 'time_band']):
    if band is None:
        continue
    n_arrivals = group['arrival_min'].notna().sum()
    hours = band_hours.get(band, 1)
    freq_records.append({
        'tract_geoid': tract,
        'time_band': band,
        'trips_per_hour': n_arrivals / hours,
    })

freq_df = pd.DataFrame(freq_records)
tract_freq = freq_df.pivot_table(index='tract_geoid', columns='time_band', 
                                  values='trips_per_hour', fill_value=0).reset_index()
tract_freq.columns = ['tract_geoid'] + [f'freq_{band}_tph' for band in tract_freq.columns[1:]]

print(f"  Tract-level frequency: {tract_freq.shape}")
for col in [c for c in tract_freq.columns if c.startswith('freq_')]:
    med = tract_freq[col].median()
    print(f"  {col}: {med:.1f} trips/hr (median)")

### 3.5 Weekend vs Weekday Service Gap

The weekend service gap measures how much transit service drops on weekends
compared to weekdays. A ratio of 1.0 means full parity; lower values mean
weekends are underserved — a problem for service workers, caregivers, and
anyone without a weekday 9-to-5 schedule.

In [ ]:
# Weekend gap: ratio of Saturday+Sunday trips to weekday trips per tract
print("Computing weekend/weekday service gap per tract...")

day_trips = st.groupby(['tract_geoid', 'service_day'])['trip_id'].count().unstack(fill_value=0)

tract_weekend = pd.DataFrame({'tract_geoid': day_trips.index})

weekday_col = day_trips['weekday'] if 'weekday' in day_trips.columns else 0
saturday_col = day_trips['saturday'] if 'saturday' in day_trips.columns else 0
sunday_col = day_trips['sunday'] if 'sunday' in day_trips.columns else 0

tract_weekend['weekday_arrivals'] = weekday_col.values
tract_weekend['saturday_arrivals'] = saturday_col.values if hasattr(saturday_col, 'values') else 0
tract_weekend['sunday_arrivals'] = sunday_col.values if hasattr(sunday_col, 'values') else 0

# [v2 FIX B] Weekend/weekday ratio: average weekend day vs ONE weekday
# GTFS service_id already represents a single day's schedule (e.g., service_id for
# "weekday" gives Monday's trips, NOT Mon-Fri combined). So weekday_arrivals is already
# per-day. No division by 5.
# Weekend per-day = (sat + sun) / 2; weekday per-day = weekday_arrivals as-is.
tract_weekend['weekend_weekday_ratio'] = np.where(
    tract_weekend['weekday_arrivals'] > 0,
    ((tract_weekend['saturday_arrivals'] + tract_weekend['sunday_arrivals']) / 2) / 
    tract_weekend['weekday_arrivals'],
    0
)

tract_weekend = tract_weekend.reset_index(drop=True)
print(f"  Weekend/weekday ratio: {tract_weekend['weekend_weekday_ratio'].median():.3f} (median)")
print(f"  Tracts with zero weekend service: {(tract_weekend['weekend_weekday_ratio'] == 0).sum()}")
print(f"  Ratio range: {tract_weekend['weekend_weekday_ratio'].min():.3f} to {tract_weekend['weekend_weekday_ratio'].max():.3f}")

### 3.6 Route Diversity and Accessibility

Route diversity captures how many different transit options serve a tract.
More routes and destinations = more flexibility for residents. We also compute
the share of wheelchair-accessible and bike-friendly trips.

In [ ]:
# Route diversity per tract (weekday)
print("Computing route diversity per tract...")

# Count unique routes and stops per tract
tract_diversity = weekday_st.groupby('tract_geoid').agg(
    unique_routes=('route_id', 'nunique'),
    unique_stops=('stop_id', 'nunique'),
).reset_index()

# [v3] Count unique headsigns (destination diversity)
# Only tracts that appear in weekday_st have transit service.
# headsign_counts will only have rows for tracts WITH service — good.
if 'trip_headsign' in weekday_st.columns:
    headsign_counts = weekday_st.groupby('tract_geoid')['trip_headsign'].nunique().reset_index()
    headsign_counts.columns = ['tract_geoid', 'unique_headsigns']
    tract_diversity = tract_diversity.merge(headsign_counts, on='tract_geoid', how='left')
    # [v3 FIX] Fill with 0, not median — tracts in this df all have service,
    # so NaN here means the headsign column was null for all trips (unlikely but safe).
    tract_diversity['unique_headsigns'] = tract_diversity['unique_headsigns'].fillna(0).astype(int)
else:
    tract_diversity['unique_headsigns'] = tract_diversity['unique_routes']

# Wheelchair accessibility (from trip-level flags)
# wheelchair_accessible: 1=yes, 2=no
# [v3] Dropped bike_friendly_pct — perfectly correlated (r=1.000) with wheelchair_pct.
# In Miami-Dade GTFS, wheelchair and bike flags are set identically across all trips.
access_df = weekday_st.groupby('tract_geoid').agg(
    total_trip_stops=('trip_id', 'count'),
    wheelchair_yes=('wheelchair_accessible', lambda x: (x == 1).sum()),
).reset_index()

access_df['wheelchair_pct'] = access_df['wheelchair_yes'] / access_df['total_trip_stops']

tract_diversity = tract_diversity.merge(
    access_df[['tract_geoid', 'wheelchair_pct']], 
    on='tract_geoid', how='left'
)

# Route type mix (bus vs rail)
route_type_map = routes.set_index('route_id')['route_type'].to_dict()
weekday_st_rt = weekday_st.copy()
weekday_st_rt['route_type'] = weekday_st_rt['route_id'].map(route_type_map)

rail_share = weekday_st_rt.groupby('tract_geoid').apply(
    lambda g: (g['route_type'].isin([0, 1, 2])).mean()
).reset_index(name='rail_trip_share')

tract_diversity = tract_diversity.merge(rail_share, on='tract_geoid', how='left')

print(f"  Route diversity: {tract_diversity.shape}")
print(f"  Median unique routes per tract: {tract_diversity['unique_routes'].median():.0f}")
print(f"  Median wheelchair %: {tract_diversity['wheelchair_pct'].median()*100:.1f}%")

### 3.7 Consolidate GTFS Features

Merge all GTFS sub-features into a single tract-level dataframe.

In [ ]:
# Merge all GTFS features
print("Consolidating GTFS features...")
gtfs_features = tract_headways.copy()

for df in [tract_span, tract_freq, tract_weekend[['tract_geoid', 'weekend_weekday_ratio']], tract_diversity]:
    gtfs_features = gtfs_features.merge(df, on='tract_geoid', how='outer')

# [v3] Collapse multicollinear time-band features.
# All 5 headway bands correlate r>0.96 with each other; all 5 freq bands correlate r>0.97.
# Keeping all of them just multiplies the same signal. We retain:
#   - peak_am: the most commonly used benchmark for transit quality
#   - early: the most distinct band (r=0.78-0.89 with others), captures overnight/pre-dawn
#     service that is equity-critical for shift workers
# Dropped: midday, peak_pm, evening (all r>0.96 with peak_am, add no unique information)
drop_redundant_bands = [
    'headway_midday_min', 'headway_peak_pm_min', 'headway_evening_min',
    'freq_midday_tph', 'freq_peak_pm_tph', 'freq_evening_tph',
    'max_service_span_hours',  # r=0.987 with mean_service_span_hours
]
existing_drops = [c for c in drop_redundant_bands if c in gtfs_features.columns]
gtfs_features = gtfs_features.drop(columns=existing_drops)
print(f"  Dropped {len(existing_drops)} multicollinear GTFS bands: {existing_drops}")

# Fill tracts with no transit service
# These tracts have NO stops in the GTFS spatial join, so all GTFS features are NaN.
headway_cols = [c for c in gtfs_features.columns if 'headway' in c]
freq_cols = [c for c in gtfs_features.columns if 'freq_' in c]
count_cols = [c for c in ['unique_routes', 'unique_stops', 'total_weekday_trips'] 
              if c in gtfs_features.columns]

for col in headway_cols:
    gtfs_features[col] = gtfs_features[col].fillna(120)  # 120 min = effectively no service
for col in freq_cols + count_cols:
    gtfs_features[col] = gtfs_features[col].fillna(0)
for col in ['mean_service_span_hours']:
    if col in gtfs_features.columns:
        gtfs_features[col] = gtfs_features[col].fillna(0)
gtfs_features['weekend_weekday_ratio'] = gtfs_features['weekend_weekday_ratio'].fillna(0)
gtfs_features['wheelchair_pct'] = gtfs_features['wheelchair_pct'].fillna(0)
gtfs_features['rail_trip_share'] = gtfs_features['rail_trip_share'].fillna(0)
# [v3 FIX] unique_headsigns: 0 for no-transit tracts, not median
gtfs_features['unique_headsigns'] = gtfs_features['unique_headsigns'].fillna(0)

print(f"\nGTFS features: {gtfs_features.shape[0]} tracts × {gtfs_features.shape[1]-1} features")
print(f"\nRetained features:")
for col in gtfs_features.columns:
    if col == 'tract_geoid':
        continue
    print(f"  {col:35s}  median={gtfs_features[col].median():8.2f}  std={gtfs_features[col].std():8.2f}")

## 4. Multi-Year ACS Trend Slopes (Sub-Sprint 2b.2)

We compute per-tract temporal trend slopes across 5 ACS vintages (2019-2023).
A positive poverty slope means poverty is rising in that tract; a negative
vehicle-access slope means fewer households have cars. These trends capture
which communities are improving or deteriorating — critical for risk scoring.

### Handling the 2019 boundary change
2019 ACS uses 2010 Census tract boundaries (519 tracts), while 2020-2023 uses
2020 boundaries (707 tracts). We apply the Census 2010→2020 tract crosswalk
with area-weighted mapping to align 2019 data to 2020 geography.

In [ ]:
# Load 2010→2020 tract crosswalk
print("Loading tract crosswalk (2010 → 2020)...")
crosswalk = pd.read_csv(CROSSWALK_PATH, sep='|', dtype=str)
print(f"  Full crosswalk: {len(crosswalk)} rows (all Florida)")

# Filter to Miami-Dade (GEOID starts with 12086)
crosswalk = crosswalk[crosswalk['GEOID_TRACT_20'].str.startswith('12086')].copy()
print(f"  Miami-Dade rows: {len(crosswalk)}")

# Parse area fields for weighting
crosswalk['AREALAND_PART'] = pd.to_numeric(crosswalk['AREALAND_PART'], errors='coerce')
crosswalk['AREALAND_TRACT_10'] = pd.to_numeric(crosswalk['AREALAND_TRACT_10'], errors='coerce')

# Weight = fraction of the 2010 tract's land area that maps to each 2020 tract
crosswalk['area_weight'] = crosswalk['AREALAND_PART'] / crosswalk['AREALAND_TRACT_10']
crosswalk['area_weight'] = crosswalk['area_weight'].clip(0, 1).fillna(0)

print(f"  Unique 2010 tracts: {crosswalk['GEOID_TRACT_10'].nunique()}")
print(f"  Unique 2020 tracts: {crosswalk['GEOID_TRACT_20'].nunique()}")
print(f"  1:1 mappings: {crosswalk.groupby('GEOID_TRACT_10').size().eq(1).sum()}")
print(f"  Splits (1→many): {crosswalk.groupby('GEOID_TRACT_10').size().gt(1).sum()}")

In [ ]:
# Map 2019 ACS data from 2010 GEOIDs to 2020 GEOIDs using area weights
print("Mapping 2019 ACS data to 2020 tract boundaries...")

acs_2019 = acs_years[2019].copy()
acs_2019 = acs_2019.rename(columns={'GEOID': 'GEOID_TRACT_10'})

# Merge with crosswalk
merged_2019 = acs_2019.merge(
    crosswalk[['GEOID_TRACT_10', 'GEOID_TRACT_20', 'area_weight']],
    on='GEOID_TRACT_10', how='inner'
)
print(f"  2019 tracts matched to crosswalk: {merged_2019['GEOID_TRACT_10'].nunique()}")

# For percentage variables: use area-weighted average when a 2010 tract splits
# For count variables (population, housing units): use area-weighted sum
numeric_cols = [c for c in acs_2019.columns 
                if c not in ['GEOID_TRACT_10', 'NAME', 'year'] 
                and acs_2019[c].dtype in ['float64', 'int64']]

# Weight the values
for col in numeric_cols:
    merged_2019[f'{col}_weighted'] = merged_2019[col] * merged_2019['area_weight']

# Aggregate to 2020 tract level
agg_dict = {f'{col}_weighted': 'sum' for col in numeric_cols}
agg_dict['area_weight'] = 'sum'  # total weight per 2020 tract (should be ~1.0)

acs_2019_mapped = merged_2019.groupby('GEOID_TRACT_20').agg(agg_dict).reset_index()

# Normalize by total weight (handles partial coverage)
for col in numeric_cols:
    acs_2019_mapped[col] = np.where(
        acs_2019_mapped['area_weight'] > 0,
        acs_2019_mapped[f'{col}_weighted'] / acs_2019_mapped['area_weight'],
        np.nan
    )

# Clean up
acs_2019_mapped = acs_2019_mapped.rename(columns={'GEOID_TRACT_20': 'GEOID'})
acs_2019_mapped = acs_2019_mapped[['GEOID'] + numeric_cols]
acs_2019_mapped['year'] = 2019

print(f"  2019 mapped to 2020 boundaries: {len(acs_2019_mapped)} tracts")
print(f"  (vs {len(acs_years[2020])} tracts in 2020-2023)")

In [ ]:
# Stack all years into a panel (2019 mapped + 2020-2023 as-is)
print("Building 5-year panel dataset...")

panel_frames = [acs_2019_mapped]
for year in [2020, 2021, 2022, 2023]:
    df = acs_years[year].copy()
    df = df.rename(columns={'GEOID': 'GEOID'})
    # Keep only numeric + identifiers
    keep_cols = ['GEOID'] + [c for c in numeric_cols if c in df.columns] + ['year']
    panel_frames.append(df[keep_cols])

panel = pd.concat(panel_frames, ignore_index=True)
panel['GEOID'] = panel['GEOID'].astype(str)

print(f"  Panel shape: {panel.shape}")
print(f"  Years: {sorted(panel['year'].unique())}")
print(f"  Tracts with all 5 years: {panel.groupby('GEOID').size().eq(5).sum()}")
print(f"  Tracts with <5 years: {panel.groupby('GEOID').size().lt(5).sum()}")

### 4.1 Compute Trend Slopes

For each tract and each ACS variable, we fit a simple linear regression
across the 5 yearly observations. The slope tells us the annual rate of
change. We require at least 3 data points to compute a trend (tracts with
fewer are assigned slope = 0, meaning "no observed trend").

In [ ]:
# Compute per-tract trend slopes via linear regression
# Variables to compute trends for (key equity-relevant metrics)
trend_vars = [
    'poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
    'rent_burden_30pct_plus', 'rent_burden_50pct_plus',
    'unemployment_rate_pct', 'median_household_income', 'per_capita_income',
    'commute_public_transit_pct', 'commute_drove_alone_pct', 'commute_wfh_pct',
    'mean_commute_time_min', 'median_gross_rent',
    'total_population', 'median_age', 'under_18_pct', 'age_65_plus_pct',
    'owner_occupied_pct', 'renter_occupied_pct',
    'no_health_insurance_pct', 'foreign_born_pct',
    'hispanic_latino_pct', 'black_alone_pct', 'white_alone_pct', 'asian_alone_pct',
]

# Filter to variables actually present in panel
trend_vars = [v for v in trend_vars if v in panel.columns]
print(f"Computing trend slopes for {len(trend_vars)} variables...")

years_array = np.array([2019, 2020, 2021, 2022, 2023], dtype=float)

def compute_slope(group, var):
    """Compute linear slope for a variable across years. Returns slope per year."""
    vals = group.set_index('year')[var].reindex([2019, 2020, 2021, 2022, 2023])
    valid = vals.dropna()
    if len(valid) < 3:
        return np.nan
    x = valid.index.values.astype(float)
    y = valid.values.astype(float)
    # Simple OLS: slope = cov(x,y) / var(x)
    slope = np.polyfit(x, y, 1)[0]
    return slope

slopes_records = []
for geoid, group in panel.groupby('GEOID'):
    record = {'GEOID': geoid}
    for var in trend_vars:
        record[f'trend_{var}'] = compute_slope(group, var)
    slopes_records.append(record)

trends_df = pd.DataFrame(slopes_records)

# Fill NaN slopes with 0 (no observed trend)
trend_cols = [c for c in trends_df.columns if c.startswith('trend_')]
trends_df[trend_cols] = trends_df[trend_cols].fillna(0)

print(f"\nTrend slopes: {trends_df.shape[0]} tracts × {len(trend_cols)} trend features")
print(f"\nTop trend signals (by median absolute slope):")
median_abs = trends_df[trend_cols].abs().median().sort_values(ascending=False)
for col in median_abs.head(10).index:
    med = trends_df[col].median()
    print(f"  {col:45s}  median slope = {med:+.4f}/year")

## 5. Feature Consolidation (Sub-Sprint 2b.3)

Merge all feature sources into a single modeling-ready dataset:
- **Sprint 2a equity indicators** (target variable + current demographics)
- **GTFS service metrics** (headways, frequency, span, diversity)
- **ACS temporal trends** (5-year slopes per variable)
- **Spatial features** (neighboring tract characteristics)
- **Interaction terms** (domain-informed feature crosses)

In [ ]:
# Start with equity indicators as the base (512 tracts = our study universe)
print("Building consolidated feature set...")

# Select relevant columns from equity indicators
equity_cols = [
    'tract_geoid',
    # Target
    'equity_priority_score', 'equity_tier', 'equity_percentile',
    # Component scores
    'composite_need', 'composite_access_deficit',
    # Individual indicators (for reference/alternative targets)
    'ind_1_transit_dependency', 'ind_2_temporal_mismatch', 'ind_3_structural_gap',
    'ind_4_time_tax', 'ind_5_service_coverage', 'ind_6_economic_vulnerability',
    'ind_7_multimodal_deficit',
    # Current demographics from v3
    'poverty_rate_pct', 'hh_no_vehicle_pct', 'snap_benefits_pct',
    'rent_burden_50pct_plus', 'unemployment_rate_pct',
    # Transit from v3
    'stop_count', 'route_count', 'transit_jobs_30_mean',
    'auto_jobs_30_mean',
    # [v3] Dropped: time_tax_minutes_mean (r=1.000 with ind_4_time_tax — exact duplicate)
    #              transit_desert (zero variance — all 0 in study universe)
    # Flags
    'institutional_tract',
]

# Keep only columns that exist
equity_cols = [c for c in equity_cols if c in equity.columns]
base = equity[equity_cols].copy()
base['tract_geoid'] = base['tract_geoid'].astype(str)
print(f"  Base (equity indicators): {base.shape}")

# Merge GTFS features
base = base.merge(gtfs_features, on='tract_geoid', how='left')
print(f"  + GTFS features: {base.shape}")

# Merge ACS trends
trends_df_renamed = trends_df.rename(columns={'GEOID': 'tract_geoid'})
trends_df_renamed['tract_geoid'] = trends_df_renamed['tract_geoid'].astype(str)
base = base.merge(trends_df_renamed, on='tract_geoid', how='left')
print(f"  + ACS trends: {base.shape}")

# Fill missing GTFS features (tracts outside GTFS coverage = no service)
gtfs_cols = [c for c in gtfs_features.columns if c != 'tract_geoid']
for col in gtfs_cols:
    if col not in base.columns:
        continue
    if 'headway' in col:
        base[col] = base[col].fillna(120)
    elif any(x in col for x in ['freq', 'trips', 'count', 'routes', 'stops', 'headsigns']):
        base[col] = base[col].fillna(0)
    elif 'span' in col:
        base[col] = base[col].fillna(0)
    elif 'ratio' in col or 'pct' in col or 'share' in col:
        base[col] = base[col].fillna(0)

# Fill missing trend features with 0 (no trend data)
for col in trend_cols:
    if col in base.columns:
        base[col] = base[col].fillna(0)

print(f"  Missing values after fill: {base.isnull().sum().sum()}")

### 5.1 Spatial Features

Compute neighboring tract characteristics. For each tract, we find all adjacent tracts
(sharing a boundary) and compute the mean equity score of neighbors. This captures
spatial spillover effects — a tract surrounded by high-equity areas may behave differently
than one surrounded by transit deserts.

In [ ]:
# Compute spatial neighbor features using tract geometries
print("Computing spatial neighbor features...")

# Reuse tract boundaries from Section 2
# Find adjacent tracts (shared boundary)
from shapely.strtree import STRtree

# Buffer tracts slightly to catch shared boundaries
tract_boundaries_proj = tract_boundaries.to_crs(epsg=3857)  # project for distance
tract_boundaries_proj['geometry_buffered'] = tract_boundaries_proj.geometry.buffer(50)  # 50m buffer

neighbor_dict = {}
for idx, row in tract_boundaries_proj.iterrows():
    tract_id = row['tract_geoid']
    # Find tracts whose buffered geometry intersects this one
    intersecting = tract_boundaries_proj[
        tract_boundaries_proj.geometry_buffered.intersects(row['geometry'])
    ]['tract_geoid'].tolist()
    # Exclude self
    neighbors = [t for t in intersecting if t != tract_id]
    neighbor_dict[tract_id] = neighbors

# Compute neighbor mean equity score
equity_score_map = base.set_index('tract_geoid')['equity_priority_score'].to_dict()

base['neighbor_mean_equity_score'] = base['tract_geoid'].apply(
    lambda t: np.nanmean([equity_score_map.get(n, np.nan) for n in neighbor_dict.get(t, [])]) 
    if len(neighbor_dict.get(t, [])) > 0 else np.nan
)

# Neighbor mean for key GTFS metrics
if 'headway_peak_am_min' in base.columns:
    headway_map = base.set_index('tract_geoid')['headway_peak_am_min'].to_dict()
    base['neighbor_mean_headway_peak'] = base['tract_geoid'].apply(
        lambda t: np.nanmean([headway_map.get(n, np.nan) for n in neighbor_dict.get(t, [])]) 
        if len(neighbor_dict.get(t, [])) > 0 else np.nan
    )

# Number of neighbors (connectivity measure)
base['n_neighbors'] = base['tract_geoid'].apply(lambda t: len(neighbor_dict.get(t, [])))

# Fill remaining NaN spatial features with column median
for col in ['neighbor_mean_equity_score', 'neighbor_mean_headway_peak']:
    if col in base.columns:
        base[col] = base[col].fillna(base[col].median())

print(f"  Median neighbors per tract: {base['n_neighbors'].median():.0f}")
print(f"  Neighbor mean equity score range: {base['neighbor_mean_equity_score'].min():.3f} to {base['neighbor_mean_equity_score'].max():.3f}")

### 5.2 Interaction Terms

Domain-informed feature interactions that capture compounding effects.
For example, high poverty combined with long headways creates a worse equity
outcome than either factor alone — the interaction captures this synergy.

In [ ]:
# Interaction terms — domain-informed crosses
print("Creating interaction terms...")

# [v3] Reduced from 6 to 4 interactions after correlation analysis:
#   Dropped poverty_x_headway (r=0.973 with poverty_x_low_span — nearly identical signal;
#     poverty_x_low_span has stronger target correlation: r=0.482 vs r=0.397)
#   Dropped rising_novehicle_x_current (r=0.050 with target — effectively zero signal)

# Poverty × low service span: high poverty + short operating hours = compounding disadvantage
if 'mean_service_span_hours' in base.columns:
    base['poverty_x_low_span'] = base['poverty_rate_pct'] * (24 - base['mean_service_span_hours'])

# No-vehicle × low service volume: transit-dependent + few trips = stranded
if 'total_weekday_trips' in base.columns:
    base['novehicle_x_low_service'] = base['hh_no_vehicle_pct'] * (1 / (1 + base['total_weekday_trips']))

# No-vehicle × weekend gap: transit-dependent + weak weekend service = weekend isolation
if 'weekend_weekday_ratio' in base.columns:
    base['novehicle_x_weekend_gap'] = base['hh_no_vehicle_pct'] * (1 - base['weekend_weekday_ratio'])

# Rising poverty × already-poor: worsening conditions in vulnerable tracts
if 'trend_poverty_rate_pct' in base.columns:
    base['rising_poverty_x_current'] = base['trend_poverty_rate_pct'].clip(lower=0) * base['poverty_rate_pct']

interaction_cols = [c for c in base.columns if '_x_' in c]
print(f"  Created {len(interaction_cols)} interaction terms:")
for col in interaction_cols:
    r = base[col].corr(base['equity_priority_score'])
    print(f"    {col:40s}  r={r:+.3f} with target")

## 6. Output and Validation

Save the final feature set and run validation checks to ensure data quality
before passing to Notebook 2 (Modeling).

In [ ]:
# Remove institutional tracts from modeling dataset
if 'institutional_tract' in base.columns:
    n_inst = base['institutional_tract'].sum()
    base_clean = base[base['institutional_tract'] == 0].copy()
    print(f"Removed {n_inst} institutional tracts")
else:
    base_clean = base.copy()

# [v2 FIX C] Deduplicate tract rows — spatial join boundary cases can produce
# duplicate stop→tract mappings that propagate through outer merges.
n_before = len(base_clean)
n_unique = base_clean['tract_geoid'].nunique()
if n_before != n_unique:
    print(f"WARNING: {n_before - n_unique} duplicate tract rows detected — deduplicating...")
    # Keep the first occurrence (all rows for same tract should be identical after groupby aggregations)
    base_clean = base_clean.drop_duplicates(subset='tract_geoid', keep='first')
    print(f"  Deduped: {n_before} → {len(base_clean)} tracts")
else:
    print(f"No duplicate tracts — {n_unique} unique tract rows")

# Identify feature columns (exclude target, identifiers, and tier labels)
target_cols = ['equity_priority_score', 'equity_tier', 'equity_percentile', 
               'composite_need', 'composite_access_deficit',
               'ind_1_transit_dependency', 'ind_2_temporal_mismatch', 'ind_3_structural_gap',
               'ind_4_time_tax', 'ind_5_service_coverage', 'ind_6_economic_vulnerability',
               'ind_7_multimodal_deficit']
id_cols = ['tract_geoid', 'institutional_tract']
feature_cols = [c for c in base_clean.columns if c not in target_cols + id_cols]

print(f"\nFinal dataset: {base_clean.shape[0]} tracts")
print(f"  Target columns: {len([c for c in target_cols if c in base_clean.columns])}")
print(f"  Feature columns: {len(feature_cols)}")
print(f"  Total columns: {base_clean.shape[1]}")

# Check for remaining NaN
nan_counts = base_clean[feature_cols].isnull().sum()
if nan_counts.sum() > 0:
    print(f"\nWARNING: {nan_counts.sum()} NaN values remain:")
    print(nan_counts[nan_counts > 0])
    # Fill remaining with column median
    for col in feature_cols:
        if base_clean[col].isnull().any():
            base_clean[col] = base_clean[col].fillna(base_clean[col].median())
    print("  → Filled with column medians")
else:
    print("  No NaN values — clean dataset!")

In [ ]:
# ======================================================================
# MODELING-READY VALIDATION (v3)
# ======================================================================
# This cell proves the output is genuinely modeling-ready:
# no bugs, no redundancy, no zero-variance, no broken fills.
print("="*70)
print("MODELING-READY VALIDATION (v3)")
print("="*70)

numeric_features = base_clean.select_dtypes(include=[np.number])
feature_only = numeric_features.drop(columns=[c for c in target_cols if c in numeric_features.columns], errors='ignore')

# --- CHECK 1: No sentinel contamination in trend slopes ---
trend_cols_check = [c for c in base_clean.columns if c.startswith('trend_')]
max_abs_slope = base_clean[trend_cols_check].abs().max().max()
print(f"\n[1] Sentinel check: max |slope| = {max_abs_slope:,.1f}")
assert max_abs_slope < 1_000_000, "FAIL — sentinel contamination"
print("    ✓ PASS")

# --- CHECK 2: Weekend/weekday ratio ---
ratio_med = base_clean['weekend_weekday_ratio'].median()
print(f"\n[2] Weekend ratio median: {ratio_med:.3f}")
assert ratio_med < 1.0, "FAIL — ratio still inflated"
print("    ✓ PASS")

# --- CHECK 3: No duplicate tracts ---
n_dupes = len(base_clean) - base_clean['tract_geoid'].nunique()
print(f"\n[3] Duplicate tracts: {n_dupes}")
assert n_dupes == 0, "FAIL — duplicates remain"
print("    ✓ PASS")

# --- CHECK 4: No zero-variance features ---
low_var = [c for c in feature_cols if base_clean[c].std() < 1e-6]
print(f"\n[4] Zero-variance features: {low_var if low_var else 'none'}")
assert len(low_var) == 0, f"FAIL — zero-variance: {low_var}"
print("    ✓ PASS")

# --- CHECK 5: No r=1.0 duplicate pairs ---
corr = feature_only.corr()
perfect_pairs = []
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i,j]) > 0.999:
            perfect_pairs.append((corr.index[i], corr.columns[j], corr.iloc[i,j]))
print(f"\n[5] Feature pairs with |r| > 0.999: {len(perfect_pairs)}")
if perfect_pairs:
    for c1, c2, r in perfect_pairs:
        print(f"    {c1} × {c2}: r={r:+.4f}")
assert len(perfect_pairs) == 0, "FAIL — duplicate features remain"
print("    ✓ PASS")

# --- CHECK 6: No median-fill artifacts for no-transit tracts ---
no_transit = base_clean['total_weekday_trips'] == 0
hs_no_transit = base_clean.loc[no_transit, 'unique_headsigns']
print(f"\n[6] unique_headsigns for no-transit tracts: {hs_no_transit.unique()}")
assert (hs_no_transit == 0).all(), "FAIL — headsigns should be 0 for no-transit tracts"
print("    ✓ PASS")

# --- CHECK 7: No NaN ---
total_nan = base_clean[feature_cols].isnull().sum().sum()
print(f"\n[7] Total NaN in features: {total_nan}")
assert total_nan == 0, "FAIL — NaN remaining"
print("    ✓ PASS")

# --- CHECK 8: Max inter-feature correlation ---
max_corr = 0
max_pair = ('', '')
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i,j]) > max_corr:
            max_corr = abs(corr.iloc[i,j])
            max_pair = (corr.index[i], corr.columns[j])
print(f"\n[8] Highest feature correlation: |r|={max_corr:.3f} ({max_pair[0]} × {max_pair[1]})")
if max_corr > 0.95:
    print(f"    ⚠ NOTE: r > 0.95 — tree models handle this; regression should use regularization")
else:
    print("    ✓ PASS — all pairs below r=0.95")

# --- SUMMARY ---
print(f"\n{'='*70}")
print(f"DATASET SUMMARY")
print(f"{'='*70}")
print(f"  Tracts:           {base_clean.shape[0]}")
print(f"  Total columns:    {base_clean.shape[1]}")
n_gtfs = len([c for c in base_clean.columns if any(x in c for x in ['headway', 'freq_', 'span', 'weekend', 'unique_routes', 'unique_stops', 'unique_headsigns', 'wheelchair', 'rail_trip', 'total_weekday'])])
n_trend = len([c for c in base_clean.columns if c.startswith('trend_')])
n_spatial = len([c for c in base_clean.columns if 'neighbor' in c or c == 'n_neighbors'])
n_interact = len([c for c in base_clean.columns if '_x_' in c])
print(f"  GTFS features:    {n_gtfs}")
print(f"  ACS trend slopes: {n_trend}")
print(f"  Spatial features: {n_spatial}")
print(f"  Interaction terms: {n_interact}")
print(f"  Target:           equity_priority_score")

# Feature-target correlations (top 15)
print(f"\nTop 15 feature-target correlations:")
target = 'equity_priority_score'
corrs = base_clean[feature_cols].corrwith(base_clean[target]).abs().sort_values(ascending=False)
for col in corrs.head(15).index:
    r = base_clean[col].corr(base_clean[target])
    print(f"  {col:45s}  r = {r:+.3f}")

In [ ]:
# Save output
OUTPUT_PATH = f'{BASE}/Sprint 2 /Sprint2b_Modeling_Features.csv'
base_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"  Shape: {base_clean.shape}")
print(f"  File size: {pd.io.common.file_exists(OUTPUT_PATH)}")

# Summary table
print(f"\n{'='*70}")
print(f"FEATURE ENGINEERING COMPLETE")
print(f"{'='*70}")
print(f"  Tracts:           {base_clean.shape[0]}")
print(f"  Total columns:    {base_clean.shape[1]}")
print(f"  GTFS features:    {len([c for c in gtfs_features.columns if c != 'tract_geoid'])}")
print(f"  ACS trend slopes: {len(trend_cols)}")
print(f"  Spatial features: {len([c for c in base_clean.columns if 'neighbor' in c or c == 'n_neighbors'])}")
print(f"  Interaction terms:{len(interaction_cols)}")
print(f"  Target:           equity_priority_score")
print(f"\nOutput file: Sprint2b_Modeling_Features.csv")
print(f"This CSV is the input to Notebook 2 (Modeling & Risk Scoring)")